# Day 9: Concurrency and async: asyncio, threads, processes, locks (Solutions)

**Beyond Python for Everybody.** Reference: the `asyncio` and `concurrent.futures` docs, and FastAPI's "Concurrency and async / await" page (fastapi.tiangolo.com/async/).

**Why it matters at ~4 years of experience:** FastAPI is async. Interviewers expect you to know when to use asyncio, threads or processes, what the GIL does, and what breaks when you call blocking code in an `async def` route.

**Note:** Jupyter already runs an event loop, so these cells use top-level `await` instead of `asyncio.run(...)`. Async checks use `await acheck(...)`.

**Time:** about 100 minutes · 7 exercises

Every check below passes. Read a solution only after you've tried the exercise.

In [ ]:
# Run this cell first. It gives you the check() helper used below.
_results = {}

def check(name, test):
    """Run a test function and print a pass/fail line with a hint."""
    try:
        test()
    except AssertionError as e:
        _results[name] = False
        print(f"❌ {name}: {e}")
    except Exception as e:
        _results[name] = False
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        _results[name] = True
        print(f"✅ {name}")

def raises(exc, fn, *args, **kwargs):
    """True if fn(*args, **kwargs) raises exc."""
    try:
        fn(*args, **kwargs)
    except exc:
        return True
    except Exception:
        return False
    return False

async def acheck(name, test):
    """Like check(), for async tests. Use it as: await acheck(name, _test)."""
    try:
        await test()
    except AssertionError as e:
        _results[name] = False
        print(f"❌ {name}: {e}")
    except Exception as e:
        _results[name] = False
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        _results[name] = True
        print(f"✅ {name}")

TOTAL = 7

def score():
    done = sum(_results.values())
    print(f"Score: {done}/{TOTAL}  " + ("🎉 all green!" if done == TOTAL else "keep going, re-run the cells you missed"))

In [ ]:
import asyncio
import threading
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

## Exercise 1: Predict the output: gather vs one after another  `warm-up`

Read the code **without running it**. What will `log` contain after `run_together` and after `run_in_turn`? Fill in both lists.

In [ ]:
async def worker(name, delay, log):
    log.append(f"{name} start")
    await asyncio.sleep(delay)
    log.append(f"{name} end")

async def run_together(log):
    await asyncio.gather(worker("A", 0.05, log), worker("B", 0.01, log))

async def run_in_turn(log):
    await worker("A", 0.05, log)
    await worker("B", 0.01, log)

# gather: both start, then B (shorter sleep) finishes first
predicted_together = ["A start", "B start", "B end", "A end"]
# awaiting one by one: A runs to the end before B starts
predicted_in_turn = ["A start", "A end", "B start", "B end"]

In [ ]:
async def _test():
    log1, log2 = [], []
    await run_together(log1)
    await run_in_turn(log2)
    assert predicted_together == log1, "with gather, both tasks start before either one ends"
    assert predicted_in_turn == log2, "awaiting one coroutine and then the other runs them in sequence"

await acheck('1. Predict the output: gather vs one after another', _test)

## Exercise 2: fetch_all_ids: run I/O concurrently with gather  `medium`

`fake_fetch(i)` takes 0.1 s (like a network call). Write `async def fetch_all_ids(ids)` that fetches all of them **concurrently** and returns the results **in the same order** as `ids`.

Ten fetches one after another would take 1 s; the check expects well under 0.5 s.

In [ ]:
async def fake_fetch(i):
    await asyncio.sleep(0.1)
    return i * 10


async def fetch_all_ids(ids):
    return await asyncio.gather(*(fake_fetch(i) for i in ids))

In [ ]:
async def _test():
    start = time.perf_counter()
    result = await fetch_all_ids(range(10))
    elapsed = time.perf_counter() - start
    assert list(result or []) == [i * 10 for i in range(10)], "return the results in the same order as ids"
    assert elapsed < 0.5, f"took {elapsed:.2f}s: the fetches ran one after another, use asyncio.gather"

await acheck('2. fetch_all_ids: run I/O concurrently with gather', _test)

## Exercise 3: process_all: cap concurrency with a Semaphore  `medium`

Unlimited `gather` over 10,000 URLs would open 10,000 connections at once and get you rate-limited. Write `async def process_all(items, handle, limit)` that runs `await handle(item)` for every item with **at most `limit` running at the same time**, and returns the results in order.

In [ ]:
async def process_all(items, handle, limit):
    sem = asyncio.Semaphore(limit)

    async def one(item):
        async with sem:
            return await handle(item)

    return await asyncio.gather(*(one(item) for item in items))

In [ ]:
async def _test():
    running = 0
    peak = 0

    async def handle(x):
        nonlocal running, peak
        running += 1
        peak = max(peak, running)
        await asyncio.sleep(0.02)
        running -= 1
        return x * 2

    result = await process_all(list(range(10)), handle, limit=3)
    assert list(result or []) == [x * 2 for x in range(10)], "return every result, in order"
    assert peak <= 3, f"{peak} ran at once, but the limit is 3: use asyncio.Semaphore"
    assert peak == 3, "the work should still run concurrently (3 at a time), not one by one"

await acheck('3. process_all: cap concurrency with a Semaphore', _test)

## Exercise 4: with_timeout: wait_for and cancellation  `medium`

Write `async def with_timeout(coro, seconds, default=None)`: return the coroutine's result, or `default` if it takes longer than `seconds`.

When the timeout hits, `asyncio.wait_for` **cancels** the slow task (it gets a `CancelledError` inside). The check verifies that the slow task really was cancelled, not left running in the background.

In [ ]:
async def with_timeout(coro, seconds, default=None):
    try:
        return await asyncio.wait_for(coro, seconds)
    except TimeoutError:
        return default

In [ ]:
async def _test():
    cancelled = []

    async def slow():
        try:
            await asyncio.sleep(1)
            return "slow result"
        except asyncio.CancelledError:
            cancelled.append(True)
            raise                      # always re-raise CancelledError

    async def fast():
        await asyncio.sleep(0.01)
        return "fast result"

    assert await with_timeout(fast(), 0.5) == "fast result", "return the result when it's fast enough"
    start = time.perf_counter()
    assert await with_timeout(slow(), 0.05, default="timeout") == "timeout", "return default on timeout"
    assert time.perf_counter() - start < 0.5, "don't wait for the slow task to finish"
    assert cancelled == [True], "the slow task should be cancelled (asyncio.wait_for does this)"

await acheck('4. with_timeout: wait_for and cancellation', _test)

## Exercise 5: Threads for blocking I/O, processes for CPU  `medium`

1. `download(url)` is **blocking** code (it uses `time.sleep`, like `requests.get`). Write `download_all(urls, max_workers=5)` with a `ThreadPoolExecutor` that returns the results in order. 10 downloads × 0.1 s with 5 threads should take about 0.2 s.
2. Fill in `best_tool`: for each task, the value is `"processes"` or `"threads or asyncio"`.

In [ ]:
def download(url):
    time.sleep(0.1)          # blocking, like requests.get
    return f"<html of {url}>"


def download_all(urls, max_workers=5):
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        return list(pool.map(download, urls))


best_tool = {
    "resize 10,000 images in pure Python": "processes",          # CPU-bound
    "call 200 HTTP APIs": "threads or asyncio",                  # waiting on the network
    "read 50 files from a network drive": "threads or asyncio",  # waiting on I/O
    "compute SHA-256 hashes over 2 GB of data, in pure Python": "processes",  # CPU-bound
}

In [ ]:
def _test():
    urls = [f"https://site/{i}" for i in range(10)]
    start = time.perf_counter()
    result = download_all(urls, max_workers=5)
    elapsed = time.perf_counter() - start
    assert result == [f"<html of {u}>" for u in urls], "return results in the order of urls (pool.map keeps order)"
    assert elapsed < 0.5, f"took {elapsed:.2f}s: run the downloads in a ThreadPoolExecutor"
    expected = {
        "resize 10,000 images in pure Python": "processes",
        "call 200 HTTP APIs": "threads or asyncio",
        "read 50 files from a network drive": "threads or asyncio",
        "compute SHA-256 hashes over 2 GB of data, in pure Python": "processes",
    }
    for task, tool in expected.items():
        assert best_tool.get(task) == tool, f"{task!r}: is it CPU-bound (processes) or waiting on I/O?"

check('5. Threads for blocking I/O, processes for CPU', _test)

## Exercise 6: Interview: fetch_all with a limit, retries and partial failures  `interview`

Write `async def fetch_all(urls, fetch, limit=3, retries=2)` that returns `{"ok": {url: result}, "failed": {url: "ErrorName: message"}}`:
- at most `limit` fetches run at once
- a `ConnectionError` is **retried** up to `retries` more times, with a short backoff (`await asyncio.sleep(0.01 * 2 ** attempt)`)
- any other exception (for example `ValueError` from bad data) is **not** retried
- one URL failing must not stop the others

This is the shape of most real integration jobs: call a partner API for many records and report what failed.

In [ ]:
async def fetch_all(urls, fetch, limit=3, retries=2):
    sem = asyncio.Semaphore(limit)
    ok, failed = {}, {}

    async def one(url):
        async with sem:
            for attempt in range(retries + 1):
                try:
                    ok[url] = await fetch(url)
                    return
                except ConnectionError as e:      # temporary: retry
                    if attempt == retries:
                        failed[url] = f"{type(e).__name__}: {e}"
                    else:
                        await asyncio.sleep(0.01 * 2 ** attempt)
                except Exception as e:            # permanent: don't retry
                    failed[url] = f"{type(e).__name__}: {e}"
                    return

    await asyncio.gather(*(one(u) for u in urls))
    return {"ok": ok, "failed": failed}

In [ ]:
async def _test():
    attempts = defaultdict(int)
    running = 0
    peak = 0

    async def fake_fetch(url):
        nonlocal running, peak
        attempts[url] += 1
        running += 1
        peak = max(peak, running)
        try:
            await asyncio.sleep(0.02)
            if url == "flaky" and attempts[url] == 1:
                raise ConnectionError("connection reset")
            if url == "down":
                raise ConnectionError("connection refused")
            if url == "bad":
                raise ValueError("bad json")
            return url.upper()
        finally:
            running -= 1

    urls = ["a", "b", "flaky", "down", "bad", "c", "d"]
    res = await fetch_all(urls, fake_fetch, limit=2, retries=2)
    assert res is not None, "return a dict with 'ok' and 'failed'"
    assert res["ok"] == {"a": "A", "b": "B", "flaky": "FLAKY", "c": "C", "d": "D"}, "every good URL (and 'flaky' after a retry) belongs in ok"
    assert set(res["failed"]) == {"down", "bad"}, "'down' and 'bad' should be in failed"
    assert res["failed"]["down"].startswith("ConnectionError"), "record failures as 'ErrorName: message'"
    assert attempts["flaky"] == 2, "a ConnectionError should be retried"
    assert attempts["down"] == 3, "1 try + 2 retries = 3 attempts, then give up"
    assert attempts["bad"] == 1, "a ValueError is permanent: don't retry it"
    assert peak <= 2, f"{peak} ran at once, but the limit is 2"

await acheck('6. Interview: fetch_all with a limit, retries and partial failures', _test)

## Exercise 7: Interview: a thread-safe counter with Lock  `interview`

`UnsafeCounter` reads the value, yields to other threads, and then writes. That's exactly what `self.value += 1` does under the hood (read, add, write). Run the cell to watch it lose increments.

Write `SafeCounter` with the same read → `time.sleep(0)` → write steps in `increment()`, but protected by a `threading.Lock` stored as `self._lock`, so 4 threads × 2000 increments always gives 8000.

In [ ]:
class UnsafeCounter:
    def __init__(self):
        self.value = 0

    def increment(self):
        current = self.value
        time.sleep(0)            # lets another thread run here, just like a real context switch
        self.value = current + 1


def run_threads(counter, n_threads=4, per_thread=2000):
    def work():
        for _ in range(per_thread):
            counter.increment()
    threads = [threading.Thread(target=work) for _ in range(n_threads)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return counter.value

print("UnsafeCounter got", run_threads(UnsafeCounter()), "of 8000")


class SafeCounter:
    def __init__(self):
        self.value = 0
        self._lock = threading.Lock()

    def increment(self):
        with self._lock:
            current = self.value
            time.sleep(0)
            self.value = current + 1

In [ ]:
def _test():
    c = SafeCounter()
    assert isinstance(getattr(c, "_lock", None), type(threading.Lock())), "store a threading.Lock() as self._lock"
    assert run_threads(c) == 8000, "increments were lost: hold the lock for the whole read-modify-write"

check('7. Interview: a thread-safe counter with Lock', _test)

## Your score

In [ ]:
score()

## Say it out loud (interview questions)

Answer each one aloud in under 60 seconds, then compare.

**You need to call 500 third-party APIs for each batch job. Threads, processes or asyncio, and why?**

The work is **waiting on the network**, not computing, so processes would waste memory. **asyncio** (with an async client like `httpx.AsyncClient`) handles hundreds of concurrent requests in one thread at low cost. Use a **Semaphore** to respect the partner's rate limit, add timeouts and retries with backoff for temporary errors, and collect per-item failures instead of failing the whole batch. If the code base is sync (`requests`), a `ThreadPoolExecutor` with 20-50 workers is the pragmatic choice.

**An `async def` FastAPI endpoint calls `requests.get()`, and under load the whole service slows down. What happened, and how do you fix it?**

`requests.get` is **blocking**. Inside `async def` it runs on the event loop thread, so while it waits, *no other request* on that worker can make progress. Fixes: switch to an async client (`await httpx.AsyncClient().get(...)`), or declare the route with plain `def` so FastAPI runs it in its threadpool, or wrap the call in `await asyncio.to_thread(...)`. The same applies to sync DB drivers, `time.sleep` and heavy CPU work.

**When would you write a FastAPI route as `def` instead of `async def`?**

Use `async def` when everything inside is awaitable (async DB driver, async HTTP client). Use plain `def` when you call **blocking** libraries (sync SQLAlchemy session, `requests`, file-heavy code). FastAPI runs `def` routes in a threadpool, so they don't block the event loop. Mixing them up is the bug: blocking calls inside `async def` hurt throughput. For CPU-heavy work, neither helps; push it to a background worker (Celery, RQ, or a process pool).

**Free-threaded Python (3.13+) can run without the GIL. What changes for your code, and what doesn't?**

In the free-threaded build, threads can run Python bytecode **in parallel**, so CPU-bound threading can finally use several cores. What doesn't change: you still need locks for shared mutable state (races were always possible, and `+=` was never atomic), and asyncio is still the right tool for lots of I/O. In practice it's still opt-in (a separate build), C extensions must support it, and single-threaded code can run slightly slower. Most web services are I/O-bound, so the day-to-day impact is small.

**You have to process a 1M-row CSV and call an API for each row. Walk me through the design.**

Stream the file (`csv.DictReader` over an open file) instead of loading it all. Process rows in **batches** (a generator that yields 500 at a time), and for each batch make concurrent API calls with a Semaphore-limited `asyncio.gather` (or the API's bulk endpoint if it has one). Retry temporary errors with backoff, write permanent failures to an error file, and record progress (a checkpoint of the last row done) so a crash can resume. Log counts and timing per batch. If it needs to run regularly, make it idempotent so a rerun doesn't double-apply rows.